In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

RANDOM_STATE = 42

1. Load the CSV file

In [ ]:
file_path = "breast_cancer_prediction.csv"
df = pd.read_csv(file_path)

2. Data Description

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe(include="all").T

Missing Values

In [ ]:
missing = df.isna().sum()
missing[missing.gt(0)].sort_values(ascending=False)

Target Distribution

In [ ]:
target_distribution = df["Cancer"].value_counts().sort_index().to_frame("count")
target_distribution["percent"] = df["Cancer"].value_counts(normalize=True).sort_index().mul(100).round(2)
target_distribution

Description Plots

In [ ]:
import matplotlib.pyplot as plt

numeric_columns = df.select_dtypes(include="number").columns.drop(["Patient_ID", "Cancer"])
df[numeric_columns].hist(bins=30, figsize=(16, 10))
plt.tight_layout()
plt.show()

Correlation Matrix

In [ ]:
corr_matrix = df.drop(columns="Patient_ID").corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f", linewidths=0.5)
plt.title("Numeric Feature Correlations")
plt.tight_layout()
plt.show()

3. Feature Selection

Drop the patient ID. Biopsy result and cancer stage give away the target, so drop those too.

In [ ]:
from sklearn.model_selection import train_test_split

target = "Cancer"
excluded_columns = ["Patient_ID", "Biopsy_Result", "Cancer_Stage"]

X = df.drop(columns=[target, *excluded_columns])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

In [ ]:
from functools import partial
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features = X.select_dtypes(include="number").columns
categorical_features = X.select_dtypes(exclude="number").columns

numeric_transformer = SimpleImputer(strategy="median")
categorical_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocessor = ColumnTransformer(transformers=[("numeric", numeric_transformer, numeric_features), ("categorical", categorical_transformer, categorical_features)])

mi_score = partial(mutual_info_classif, random_state=RANDOM_STATE)
pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("selector", SelectKBest(score_func=mi_score, k="all")), ("model", HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE))])

4. Baseline Model

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"roc_auc": "roc_auc", "average_precision": "average_precision", "balanced_accuracy": "balanced_accuracy"}

baseline_scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
pd.DataFrame(baseline_scores).filter(like="test_").agg(["mean", "std"]).T

5. Hyperparameter Tuning

In [ ]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV

# parameter ranges
param_distributions = {
    "selector__k": [10, 15, 20, "all"],
    "model__learning_rate": loguniform(0.02, 0.20),
    "model__max_iter": randint(100, 401),
    "model__max_leaf_nodes": randint(15, 64),
    "model__max_depth": [None, 3, 5, 7],
    "model__min_samples_leaf": randint(10, 81),
    "model__l2_regularization": loguniform(1e-3, 10),
}

# randomized search is faster than checking every combination
search = RandomizedSearchCV(pipeline, param_distributions, n_iter=30, scoring="roc_auc", cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=1)
search.fit(X_train, y_train)

print(f"Best CV ROC-AUC: {search.best_score_:.4f}")
print("Best Parameters:")
search.best_params_

Selected Features

In [ ]:
best_model = search.best_estimator_
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
selector = best_model.named_steps["selector"]

feature_scores = pd.Series(selector.scores_, index=feature_names, name="mutual_information")
selected_features = feature_scores[selector.get_support()].sort_values(ascending=False)
selected_features

6. Test and Validate

In [ ]:
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, roc_auc_score

y_pred = best_model.predict(X_test)
y_probability = best_model.predict_proba(X_test)[:, 1]

test_metrics = pd.Series({"accuracy": accuracy_score(y_test, y_pred), "balanced_accuracy": balanced_accuracy_score(y_test, y_pred), "roc_auc": roc_auc_score(y_test, y_probability), "average_precision": average_precision_score(y_test, y_probability)}, name="test_score").round(4)
test_metrics

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

print(classification_report(y_test, y_pred, digits=4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Tuned HGBC Confusion Matrix")
plt.show()

In [ ]:
validation_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")[["rank_test_score", "mean_test_score", "std_test_score", "params"]].head(10)
validation_results

Experimental Results Plots

In [ ]:
# top tuning results
top_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score").head(10).sort_values("mean_test_score")
plt.figure(figsize=(8, 5))
plt.barh(range(1, 11), top_results["mean_test_score"], xerr=top_results["std_test_score"], color="steelblue")
plt.yticks(range(1, 11), top_results["rank_test_score"].astype(int))
plt.xlabel("Mean CV ROC-AUC")
plt.ylabel("Search rank")
plt.title("Top Hyperparameter Results")
plt.show()

In [ ]:
# strongest selected features
selected_features.head(12).sort_values().plot(kind="barh", figsize=(8, 5), color="darkcyan")
plt.xlabel("Mutual information")
plt.title("Most Informative Selected Features")
plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions(y_test, y_probability, name="Tuned HGBC")
plt.plot([0, 1], [0, 1], "k--")
plt.title("Test ROC Curve")
plt.show()

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

PrecisionRecallDisplay.from_predictions(y_test, y_probability, name="Tuned HGBC")
plt.axhline(y_test.mean(), color="black", linestyle="--", label="Cancer rate")
plt.title("Test Precision-Recall Curve")
plt.legend()
plt.show()

7. Save Model

In [ ]:
from pathlib import Path
import joblib

Path("models").mkdir(exist_ok=True)
joblib.dump(best_model, "models/breast_cancer_hgbc.joblib")